Joins the station/kitchen data with the foot traffic data to determine which floors are covered/uncovered by a water source, then rank each of these by average foot traffic. This also takes care of various conversions between floor characters ("B" for basement, "L" for lower level, etc.).

In [72]:
import pandas as pd

In [73]:
stations_df = pd.read_csv('Stations.csv')

In [74]:
kitchens_df = pd.read_csv('Kitchens.csv')

In [75]:
stations_df.head()

,Floor,Building Description,Space@Bu room,Quantity,Date Installed,Type
0,1,1 UNIVERSITY ROAD,190A,1,NaN,NaN
1,1,1 UNIVERSITY ROAD,NaN,1,NaN,NaN
2,3,1 UNIVERSITY ROAD,390,1,NaN,NaN
3,2,1 UNIVERSITY ROAD,284,1,NaN,NaN
4,1,100 ASHFORD STREET,191,1,NaN,Combo Bottle Filler/Drinking Fountain


In [76]:
kitchens_df.head()

,Floor,Building Code,Building Description,Room #
0,1,623,1 UNIVERSITY ROAD,104
1,1,623,1 UNIVERSITY ROAD,150G
2,1,623,1 UNIVERSITY ROAD,150K
3,10,500,10 BUICK STREET,1001
4,5,500,10 BUICK STREET,505


In [77]:
# track buildings and floors that have either a kitchen or bottle station.
# the set stores tuples of the form (f, b), where f is a floor and b is a building description
covered_building_floors = set()

In [78]:
# add all stations first
for f, b in zip(stations_df['Floor'], stations_df['Building Description']):
    covered_building_floors.add((str(f), b))

In [79]:
# then add all kitchens
for f, b in zip(kitchens_df['Floor'], kitchens_df['Building Description']):
    covered_building_floors.add((str(f), b))

In [80]:
# it seems the building addresses contain both ST and STREET, AVE and AVENUE, RD and ROAD...

In [81]:
# get the foot traffic table
foot_traffic_df = pd.read_csv('wifi_update.csv')

In [83]:
# create mapping for the floors in the foot traffic data
floor_mapping = {
    'B': -1, 'b': -1, 'g': 1, 'l': -1, 'm': 1
}

In [84]:
foot_traffic_df['building_floor'] = foot_traffic_df['building_floor'].map(lambda x: 
                                                                          x if x not in floor_mapping else floor_mapping[x])

In [88]:
# create a new column that has both the floor and building desc
foot_traffic_df['F,B'] = list(zip(foot_traffic_df['building_floor'].astype(str), foot_traffic_df['building_desc']))

In [97]:
# join with ventilation data
building_and_kitchen = pd.read_csv("Building and Kitchen List.csv")[['Building Description', 'Mechanical Ventilation']]

In [100]:
foot_traffic_df = pd.merge(foot_traffic_df, building_and_kitchen, how = 'left', left_on = 'building_desc', right_on = 'Building Description').drop(columns=['Building Description'])

In [87]:
# handle foot traffic address that have hyphens, representing a (contiguous?) range of addresses
def is_covered(floor, address):
    if (floor, address) in covered_building_floors:
        return True
    if '-' not in address:
        return (floor, address) in covered_building_floors
    l,r = address.split('-')
    r, street = r.split()[0], ' '.join(r.split()[1:])
    l,r = int(l), int(r)
    for i in range(l, r+1):
        if (floor, f'{i} {street}') in covered_building_floors:
            return True
    return False

In [101]:
# filter out rows that have a (floor, description) in the station/kitchen set
foot_traffic_not_covered_df = foot_traffic_df[~foot_traffic_df['F,B'].map(lambda x: is_covered(*x))]
foot_traffic_not_covered_df.drop(columns=['F,B'], inplace=True)
# filter for rows that are covered
foot_traffic_covered_df = foot_traffic_df[foot_traffic_df['F,B'].map(lambda x: is_covered(*x))]
foot_traffic_covered_df.drop(columns=['F,B'], inplace=True)

/var/folders/rz/znvx90ld6vnbmf6g3sf73y2w0000gn/T/ipykernel_6742/2542086785.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foot_traffic_not_covered_df.drop(columns=['F,B'], inplace=True)
/var/folders/rz/znvx90ld6vnbmf6g3sf73y2w0000gn/T/ipykernel_6742/2542086785.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foot_traffic_covered_df.drop(columns=['F,B'], inplace=True)


In [106]:
# also filter out parking lots from the candidates
foot_traffic_not_covered_df = foot_traffic_not_covered_df[foot_traffic_not_covered_df['building_floor']!='p']

In [108]:
# sort by mean_density and export
foot_traffic_not_covered_df.sort_values('mean_density_cnt', ascending=False, inplace=True)
foot_traffic_not_covered_df.to_csv('covered_floors.csv', index=False)

foot_traffic_covered_df.sort_values('mean_density_cnt', ascending=False, inplace=True)
foot_traffic_covered_df.to_csv('uncovered_floors_withoutvolumne.csv', index=False)

/var/folders/rz/znvx90ld6vnbmf6g3sf73y2w0000gn/T/ipykernel_6742/946144973.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foot_traffic_not_covered_df.sort_values('mean_density_cnt', ascending=False, inplace=True)
/var/folders/rz/znvx90ld6vnbmf6g3sf73y2w0000gn/T/ipykernel_6742/946144973.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foot_traffic_covered_df.sort_values('mean_density_cnt', ascending=False, inplace=True)
